# El Niño 2026 — Notebook 06: Heat budget v2 (feedback decomposition, eddies, satellite check)
**le-gis.com climate project**

Notebook 04's box budget left a large unexplained residual (+2.4 °C in Niño-3, +3.8 °C at the coast, March–September 2026). This notebook replaces it with the standard **mixed-layer heat budget with feedback decomposition** used in ENSO research, and adds two checks.

| Part | What | Why | Data |
|---|---|---|---|
| **A** | Mixed-layer (0–H m) temperature budget on a 0.5° grid; every advection term split into *normal × anomaly* parts: **thermocline feedback** −w̄ ∂T′/∂z, **Ekman (upwelling) feedback** −w′ ∂T̄/∂z, **zonal advective feedback** −u′ ∂T̄/∂x, mean-current advection, meridional, nonlinear, surface-flux damping | Answers "thermocline or weaker upwelling?" directly, in the form reviewers expect (Jin et al. 2006; Ren & Jin 2013) | GLORYS12 monthly T, u, v, mixed-layer depth (1993–now); ERA5 surface fluxes |
| **B** | Eddy heat-flux convergence −∇·(u″T″) from **daily** fields (tropical instability waves) for the four El Niño years and three neutral years | Monthly means hide the eddies; they are a main reason the old residual was large | GLORYS12 daily, March–September only |
| **C** | Zonal and meridional advection terms recomputed from **satellite** data only | Independent check of the largest warming term | Copernicus GlobCurrent (altimetry + Ekman) currents at 15 m; NOAA OISST |

**Budget (anomaly form, °C per month):**
∂T′/∂t = −u′∂T̄/∂x − ū∂T′/∂x − v′∂T̄/∂y − v̄∂T′/∂y − w̄∂T′/∂z − w′∂T̄/∂z + nonlinear + Q′/(ρc_pH) + eddy + residual,
with T the 0–H m mean, ∂T/∂z ≈ (T − T_sub)/H (T_sub at H + 10 m), upwelling w at the layer base from continuity (only upward w counts), and the 1993–2020 climatology as "normal".

**Runtime, first run:** A ≈ 1.5–2 h (GLORYS monthly, 1993–now); B ≈ 1.5–3 h (daily, 49 months); C ≈ 15–30 min. Every year/month is cached to Drive, so the notebook can be stopped and resumed.

In [ ]:
!pip -q install copernicusmarine netCDF4 earthengine-api
import os, time, getpass, logging, warnings
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
for n in ("copernicusmarine", "urllib3.connectionpool"): logging.getLogger(n).setLevel(logging.ERROR)
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "font.size": 9})

from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/le-gis_enso"
CACHE, WEB = f"{ROOT}/cache", f"{ROOT}/web"
C6, OUT = f"{CACHE}/nb06", f"{WEB}/nb06"; [os.makedirs(d, exist_ok=True) for d in (CACHE, WEB, C6, OUT)]
TODAY = pd.Timestamp.today().normalize(); this_year = TODAY.year

# ---- knobs ----
Y0, CLIM = 1993, (1993, 2020)                 # record start; climatology ("normal")
H, HS = 50, (30, 50, 70)                      # mixed-layer depth used (m) and sensitivity range
ZMAX = 100                                    # deepest level downloaded for the monthly fields (m)
EVENTS = [1997, 2015, 2023, 2026]
NEUTRAL = [2001, 2012, 2013]                  # ENSO-neutral years: reference for the eddy term
SEASON = [3, 4, 5, 6, 7, 8, 9]                # March-September, as in Notebook 04
DAY_STEP = 2                                  # daily fields: use every 2nd day (eddies have 17-33 day periods)
RES = 0.5
LATS = np.round(np.arange(-10.75, 5.76, RES), 2); LONS = np.round(np.arange(209.25, 280.76, RES), 2)   # target grid
BOXES = {"Niño-3": dict(lat=(-5, 5), lon=(210, 270)), "Niño-1+2 (coast)": dict(lat=(-10, 0), lon=(270, 280))}
COLORS = {1997: "#1f77b4", 2015: "#2ca02c", 2023: "#9467bd", 2026: "crimson"}
R_EARTH, RHO0, CP0, SEC_MONTH = 6.371e6, 1025.0, 3990.0, 30.4375 * 86400

In [ ]:
# --- logins: Copernicus Marine (GLORYS + satellite currents) and Earth Engine (ERA5 fluxes, OISST) ---
import copernicusmarine as cm, ee
# log in once per Colab session: the toolbox saves the credentials, so later downloads do not ask again
cm.login(username=input("Copernicus Marine username: "), password=getpass.getpass("Copernicus Marine password: "), force_overwrite=True)
EE_PROJECT = "YOUR-EE-PROJECT"   # <-- same Earth Engine project as Notebooks 02-05
ee.Authenticate(); ee.Initialize(project=EE_PROJECT)

## Shared tools: product coverage, download + regrid to 0.5°, derivatives, layer means

In [ ]:
def month_start(t): return pd.to_datetime(np.atleast_1d(t)).to_period("M").to_timestamp()
def clean_time(x):
    x = x.sortby("time"); _, k = np.unique(x.time.values, return_index=True); return x.isel(time=np.sort(k))
def with_retry(f, tries=4, wait=20):
    for k in range(tries):
        try: return f()
        except Exception as e:
            if k == tries - 1: raise
            print("   retry:", str(e)[:90]); time.sleep(wait * (k + 1))

# GLORYS12 dataset ids: reanalysis (my, incl. interim) first, then the real-time analysis (anfc) for the latest months
ID = {("M", "my", v): "cmems_mod_glo_phy_my_0.083deg_P1M-m" for v in ("thetao", "uo", "vo", "mlotst")}
ID.update({("M", "anfc", "thetao"): "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1M-m", ("M", "anfc", "uo"): "cmems_mod_glo_phy-cur_anfc_0.083deg_P1M-m",
           ("M", "anfc", "vo"): "cmems_mod_glo_phy-cur_anfc_0.083deg_P1M-m", ("M", "anfc", "mlotst"): "cmems_mod_glo_phy_anfc_0.083deg_P1M-m"})
ID.update({("D", "my", v): "cmems_mod_glo_phy_my_0.083deg_P1D-m" for v in ("thetao", "uo", "vo")})
ID.update({("D", "anfc", "thetao"): "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m", ("D", "anfc", "uo"): "cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
           ("D", "anfc", "vo"): "cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m"})
REG = dict(minimum_longitude=-151.0, maximum_longitude=-79.0, minimum_latitude=-11.0, maximum_latitude=6.0)   # 209-281 E, 11 S-6 N

def cm_open(ds_id, var, t0, t1, depth=None):
    kw = dict(dataset_id=ds_id, variables=[var], start_datetime=str(pd.Timestamp(t0).date()), end_datetime=str(pd.Timestamp(t1).date()), **REG)
    if depth: kw.update(minimum_depth=depth[0], maximum_depth=depth[1])
    return cm.open_dataset(**kw)[var]

COVER = {}
for f_ in ("M", "D"):
    for key in ("my", "anfc"):
        try:
            t = pd.to_datetime(cm_open(ID[(f_, key, "thetao")], "thetao", "1990-01-01", TODAY, depth=(0, 1)).time.values)
            COVER[(f_, key)] = (t.min().normalize(), t.max().normalize())
        except Exception as e: print("coverage", f_, key, "unavailable:", str(e)[:80])
print({k: f"{a:%Y-%m-%d} -> {b:%Y-%m-%d}" for k, (a, b) in COVER.items()})

def products_for(times, f_):
    out, rest = [], pd.DatetimeIndex(times)
    for key in ("my", "anfc"):
        if (f_, key) not in COVER: continue
        a, b = COVER[(f_, key)]; sel = rest[(rest >= a) & (rest <= b)]
        if len(sel): out.append((key, sel)); rest = rest.difference(sel)
    return out

def regrid(da):
    # 1/12 deg -> block mean to ~0.5 deg -> interpolate onto the common target grid (LATS, LONS)
    da = da.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items() if k in da.dims})
    da = da.assign_coords(lon=da.lon % 360).sortby("lon").sortby("lat")
    n = max(1, int(round(RES / float(np.median(np.diff(da.lon.values))))))
    if n > 1: da = da.coarsen(lat=n, lon=n, boundary="trim").mean()
    da = da.interp(lat=LATS, lon=LONS)
    return da.drop_vars([c for c in da.coords if c not in da.dims]).astype("float32")

COSL = xr.DataArray(np.cos(np.deg2rad(LATS)), dims="lat", coords={"lat": LATS})
ddx = lambda f: f.differentiate("lon") / (R_EARTH * np.deg2rad(1.0) * COSL)
ddy = lambda f: f.differentiate("lat") / (R_EARTH * np.deg2rad(1.0))

def edges_weights(z, Hh):
    e = np.concatenate([[0], 0.5 * (z[1:] + z[:-1]), [z[-1] + (z[-1] - z[-2]) / 2]])
    return np.clip(np.minimum(e[1:], Hh) - e[:-1], 0, None)
def layer(da, Hh):
    w = xr.DataArray(edges_weights(da.depth.values, Hh), dims="depth", coords={"depth": da.depth.values})
    return (da * w).sum("depth") / (w * da.notnull()).sum("depth")

def box_mean(da, name):
    b = BOXES[name]; d = da.sel(lat=slice(*b["lat"]), lon=slice(*b["lon"]))
    w = (COSL.sel(lat=d.lat) * d.notnull())
    return ((d.fillna(0) * w).sum(["lat", "lon"]) / w.sum(["lat", "lon"])).to_series()
def clim(x): return x.sel(time=slice(str(CLIM[0]), str(CLIM[1]))).groupby("time.month").mean()
def anom(x): return (x.groupby("time.month") - clim(x)).drop_vars("month")

## A1. Data — GLORYS12 monthly T, u, v (0–100 m) and mixed-layer depth, 1993–now, regridded to 0.5°

In [ ]:
def glorys_year(y):
    f = f"{C6}/glorys_box3d_{y}.nc"
    if os.path.exists(f) and y <= this_year - 2: return xr.open_dataset(f).load()
    months = pd.date_range(f"{y}-01-01", f"{y}-12-01", freq="MS"); months = months[months <= TODAY]
    parts = []
    for key, sel in products_for(months, "M"):
        t1 = sel[-1] + pd.offsets.MonthEnd(0); dv = {}
        for v in ("thetao", "uo", "vo", "mlotst"):
            dz = None if v == "mlotst" else (0, ZMAX)
            da = with_retry(lambda: regrid(cm_open(ID[("M", key, v)], v, sel[0], t1, depth=dz).load()))
            da["time"] = month_start(da.time.values); dv[v] = da
        parts.append(xr.Dataset(dv))
    ds = clean_time(xr.concat(parts, "time"))
    if y <= this_year - 2: ds.to_netcdf(f)
    return ds

G = []
for y in range(Y0, this_year + 1):
    t = time.time(); G.append(glorys_year(y))
    if time.time() - t > 3: print(f"  GLORYS {y}: {G[-1].sizes['time']} months ({time.time() - t:.0f} s)")
G = clean_time(xr.concat(G, "time"))
print("GLORYS:", dict(G.sizes), str(G.time.values[0])[:7], "->", str(G.time.values[-1])[:7])

## A2. Data — ERA5 surface heat fluxes (Earth Engine), on the same grid

In [ ]:
ERA5H = ee.ImageCollection("ECMWF/ERA5/HOURLY")
FLX = {"mean_surface_net_short_wave_radiation_flux": "sw", "mean_surface_net_long_wave_radiation_flux": "lw",
       "mean_surface_sensible_heat_flux": "sh", "mean_surface_latent_heat_flux": "lh"}
def ee_grid(img, bands):
    a = ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": len(LONS), "height": len(LATS)},
                 "affineTransform": {"scaleX": RES, "shearX": 0, "translateX": float(LONS[0] - 360 - RES / 2),
                                     "shearY": 0, "scaleY": -RES, "translateY": float(LATS[-1] + RES / 2)}, "crsCode": "EPSG:4326"}})
    out = {}
    for b in bands:
        x = np.asarray(a[b], dtype=float)[::-1]; x[x <= -9998] = np.nan; out[b] = x       # flip rows: south -> north
    return out
def flux_month(t0):
    t1 = t0 + pd.offsets.MonthBegin(1)
    for step in (1, 3, 6):
        col = ERA5H.filterDate(str(t0.date()), str(t1.date())).select(list(FLX))
        if step > 1: col = col.filter(ee.Filter.Or(*[ee.Filter.calendarRange(h, h, "hour") for h in range(0, 24, step)]))
        try: return ee_grid(col.mean().unmask(-9999, False).toFloat(), FLX)
        except Exception as e:
            if "memory" not in str(e).lower() and "timed out" not in str(e).lower(): time.sleep(10)
    raise RuntimeError(f"ERA5 fluxes {t0:%Y-%m} failed")
def flux_year(y):
    f = f"{C6}/era5_flux_{y}.nc"
    if os.path.exists(f) and y <= this_year - 1: return xr.open_dataset(f).load()
    months = [m for m in pd.date_range(f"{y}-01-01", f"{y}-12-01", freq="MS") if m <= TODAY - pd.DateOffset(days=40)]
    with ThreadPoolExecutor(max_workers=10) as ex: res = list(ex.map(flux_month, months))
    ds = xr.Dataset({s: (("time", "lat", "lon"), np.stack([r[b] for r in res])) for b, s in FLX.items()},
                    coords={"time": pd.DatetimeIndex(months), "lat": LATS, "lon": LONS})
    if y <= this_year - 1: ds.to_netcdf(f)
    return ds
FL = clean_time(xr.concat([flux_year(y) for y in range(Y0, this_year + 1)], "time"))
print("ERA5 fluxes:", str(FL.time.values[0])[:7], "->", str(FL.time.values[-1])[:7])

## A3. Budget and feedback decomposition
For each grid cell and month (then averaged over the two boxes):
* layer means T, u, v over 0–H m; T_sub at H + 10 m; w at the base from continuity, w = H·⟨∂u/∂x + ∂v/∂y⟩ (positive up, only upward w counts);
* anomalies (′) and normals (‾) relative to the 1993–2020 calendar-month climatology;
* each advection anomaly split exactly into linear parts + a nonlinear remainder.

In [ ]:
def budget(Hh, g=None, fl=None):
    g = G if g is None else g; fl = FL if fl is None else fl
    t = np.intersect1d(g.time.values, fl.time.values); g = g.sel(time=t); fl = fl.sel(time=t)
    T, U, V = layer(g.thetao, Hh), layer(g.uo, Hh), layer(g.vo, Hh)
    Ts = g.thetao.interp(depth=Hh + 10)
    W = Hh * layer(ddx(g.uo) + ddy(g.vo), Hh)                                  # m/s, upward at the layer base
    pen = 0.58 * np.exp(-Hh / 0.35) + 0.42 * np.exp(-Hh / 23.0)                # Paulson & Simpson (1977), type I water
    Q = (fl.sw + fl.lw + fl.sh + fl.lh) - pen * fl.sw
    Tc, Uc, Vc, Wc, Tsc = [clim(x) for x in (T, U, V, W, Ts)]
    Ta, Ua, Va, Tsa = anom(T), anom(U), anom(V), anom(Ts)
    cm_ = lambda c: c.sel(month=T["time.month"]).drop_vars("month")             # climatology on the time axis
    Tbar, Ubar, Vbar, Wbar, Tsbar = cm_(Tc), cm_(Uc), cm_(Vc), cm_(Wc), cm_(Tsc)
    Mw, Mwb = W.clip(min=0), Wbar.clip(min=0)
    k = SEC_MONTH
    Ax = -U * ddx(T); Ay = -V * ddy(T); Vt = -Mw * (T - Ts) / Hh                    # full advection terms
    terms = {
        "zonal advective feedback (−u′∂T̄/∂x)": -Ua * ddx(Tbar) * k,
        "mean-current advection (−ū∂T′/∂x − v̄∂T′/∂y)": (-Ubar * ddx(Ta) - Vbar * ddy(Ta)) * k,
        "meridional advection (−v′∂T̄/∂y)": -Va * ddy(Tbar) * k,
        "thermocline feedback (−w̄∂T′/∂z)": -Mwb * (Ta - Tsa) / Hh * k,
        "Ekman / upwelling feedback (−w′∂T̄/∂z)": -(Mw - Mwb) * (Tbar - Tsbar) / Hh * k,
        "surface flux (Q′/ρc_pH)": anom(Q) / (RHO0 * CP0 * Hh) * k,
    }
    lin = sum(v for kk, v in terms.items() if "surface" not in kk)
    terms["nonlinear"] = (anom(Ax) + anom(Ay) + anom(Vt)) * k - lin
    tend = (Ta.shift(time=-1) - Ta.shift(time=1)) / 2                                   # centred difference
    tend = tend.where(tend.notnull(), Ta - Ta.shift(time=1))                           # last available month: backward difference
    out = {}
    for name in BOXES:
        df = pd.DataFrame({kk: box_mean(v, name) for kk, v in terms.items()})
        df["tendency"] = box_mean(tend, name); df["T anomaly (°C)"] = box_mean(Ta, name)
        df["residual"] = df["tendency"] - df[list(terms)].sum(axis=1)
        df["w (m/day)"] = box_mean(W, name) * 86400; df["MLD (m)"] = box_mean(g.mlotst, name) if "mlotst" in g else np.nan
        out[name] = df
    return out

TERMS = None
BUD = {Hh: budget(Hh) for Hh in HS}
TERMS = [c for c in BUD[H]["Niño-3"].columns if c not in ("tendency", "T anomaly (°C)", "residual", "w (m/day)", "MLD (m)")]
for name in BOXES:
    BUD[H][name].round(4).to_csv(f"{OUT}/A_budget_monthly_{name.split()[0]}_H{H}.csv")
    print(f"{name}: mean MLD {BUD[H][name]['MLD (m)'].mean():.0f} m, mean upwelling {BUD[H][name]['w (m/day)'].mean():.2f} m/day")

In [ ]:
def event_sums(df, y, cols):
    s = df[(df.index.year == y) & (df.index.month.isin(SEASON))]
    if len(s) < 3: return None
    r = s[cols + ["tendency", "residual"]].sum(); r["months"] = len(s); return r
ROWS = []
for name in BOXES:
    for y in EVENTS:
        for Hh in HS:
            r = event_sums(BUD[Hh][name], y, TERMS)
            if r is not None: ROWS.append({"box": name, "event": y, "H": Hh, **r.to_dict()})
EVS = pd.DataFrame(ROWS); EVS.round(3).to_csv(f"{OUT}/A_event_sums.csv", index=False)
main = EVS[EVS.H == H].set_index(["box", "event"])
main["closure_%"] = 100 * (1 - main.residual.abs() / (main.residual.abs() + main[TERMS].abs().sum(axis=1)))
display(main[TERMS + ["residual", "tendency", "closure_%"]].round(2))

## B. Eddy heat-flux convergence from daily fields (tropical instability waves)

In [ ]:
def eddy_month(y, m):
    f = f"{C6}/eddy_H{H}_{y}_{m:02d}.nc"
    if os.path.exists(f): return xr.open_dataarray(f).load()
    t0 = pd.Timestamp(y, m, 1); t1 = t0 + pd.offsets.MonthEnd(0)
    days = pd.date_range(t0, t1, freq="D"); prods = products_for(days, "D")
    if not prods: return None
    key, sel = max(prods, key=lambda p: len(p[1]))
    f3 = {}
    for v in ("thetao", "uo", "vo"):
        def get():
            da = cm_open(ID[("D", key, v)], v, sel[0], sel[-1], depth=(0, H + 10))
            return regrid(da.isel(time=slice(None, None, DAY_STEP)).load())
        f3[v] = with_retry(get)
    T, U, V = (layer(f3[v], H) for v in ("thetao", "uo", "vo"))
    Tp, Up, Vp = T - T.mean("time"), U - U.mean("time"), V - V.mean("time")
    E = -(ddx((Up * Tp).mean("time")) + ddy((Vp * Tp).mean("time"))) * SEC_MONTH
    E = E.rename("eddy"); E.to_netcdf(f); return E

YB = sorted(set(EVENTS + NEUTRAL)); EDDY = []
for y in YB:
    for m in SEASON:
        if pd.Timestamp(y, m, 1) > TODAY - pd.DateOffset(days=5): continue
        t = time.time(); E = eddy_month(y, m)
        if E is None: continue
        for name in BOXES: EDDY.append({"year": y, "month": m, "box": name, "eddy": float(box_mean(E.expand_dims(time=[pd.Timestamp(y, m, 1)]), name).iloc[0])})
        if time.time() - t > 3: print(f"  daily {y}-{m:02d} ({time.time() - t:.0f} s)")
EDDY = pd.DataFrame(EDDY); EDDY.round(4).to_csv(f"{OUT}/B_eddy_monthly.csv", index=False)
ref = EDDY[EDDY.year.isin(NEUTRAL)].groupby(["box", "month"]).eddy.mean()
EDDY["eddy_anom"] = EDDY.eddy - ref.reindex(pd.MultiIndex.from_frame(EDDY[["box", "month"]])).values
EB = EDDY[EDDY.year.isin(EVENTS)].groupby(["box", "year"]).eddy_anom.sum().rename("eddy′ (°C)")
main = main.join(EB.rename_axis(["box", "event"]))
main["residual after eddy"] = main.residual - main["eddy′ (°C)"].fillna(0)
main["closure_after_%"] = 100 * (1 - main["residual after eddy"].abs() / (main["residual after eddy"].abs() + main[TERMS].abs().sum(axis=1) + main["eddy′ (°C)"].abs().fillna(0)))
main.round(3).to_csv(f"{OUT}/AB_event_budget.csv")
display(main[["eddy′ (°C)", "residual", "residual after eddy", "closure_%", "closure_after_%"]].round(2))

In [ ]:
# Figure: feedback decomposition for each event, H = 50 m, with the eddy term; error bars = range over H = 30-70 m
SHORT = {TERMS[0]: "zonal advective fb", TERMS[1]: "mean-current adv.", TERMS[2]: "meridional adv.", TERMS[3]: "thermocline fb",
         TERMS[4]: "Ekman (upwelling) fb", TERMS[5]: "surface flux", "nonlinear": "nonlinear", "eddy′ (°C)": "eddies (TIW)", "residual after eddy": "residual"}
PAL = ["#dc2626", "#f472b6", "#16a34a", "#7c3aed", "#0ea5e9", "#f59e0b", "#94a3b8", "#0f766e", "#cbd5e1"]
cols = TERMS + ["eddy′ (°C)", "residual after eddy"]
fig, axs = plt.subplots(1, 2, figsize=(16, 5.2))
for ax, name in zip(axs, BOXES):
    d = main.loc[name].reindex(EVENTS).dropna(how="all")
    x = np.arange(len(d)); pos = np.zeros(len(d)); neg = np.zeros(len(d))
    for c, colr in zip(cols, PAL):
        v = d[c].fillna(0).values; base = np.where(v >= 0, pos, neg)
        ax.bar(x, v, .6, bottom=base, color=colr, label=SHORT.get(c, c), edgecolor="white", lw=.5)
        pos += np.where(v >= 0, v, 0); neg += np.where(v < 0, v, 0)
    ax.scatter(x, d.tendency, marker="D", c="k", zorder=5, label="observed change")
    rng = EVS[EVS.box == name].pivot(index="event", columns="H", values="residual").reindex(d.index)
    ax.axhline(0, c="k", lw=.8); ax.set_xticks(x); ax.set_xticklabels([f"{e}" + (" (to date)" if e == this_year else "") for e in d.index])
    ax.set_title(f"What warmed {name}? March → September (anomaly, °C)", loc="left", fontweight="bold")
h_, l_ = axs[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="lower center", ncol=10, fontsize=8, frameon=False)
plt.tight_layout(rect=(0, .07, 1, 1))
for ext in ("png", "pdf"): plt.savefig(f"{OUT}/AB_feedback_budget.{ext}", dpi=300, bbox_inches="tight")
plt.show()

## C. Satellite check — advection from satellite currents (GlobCurrent, 15 m) and satellite SST (OISST)

In [ ]:
def find_ids(product_id, must=("P1M",)):
    try:
        cat = cm.describe(product_id=product_id)
        prods = cat.products if hasattr(cat, "products") else cat["products"]
        ids = [(d.dataset_id if hasattr(d, "dataset_id") else d["dataset_id"]) for p in prods for d in (p.datasets if hasattr(p, "datasets") else p["datasets"])]
        return [i for i in ids if all(m in i for m in must)]
    except Exception as e:
        print("describe failed:", str(e)[:80]); return []
SAT_IDS = find_ids("MULTIOBS_GLO_PHY_MYNRT_015_004") or ["cmems_obs-mob_glo_phy-cur_my_0.25deg_P1M-m", "cmems_obs-mob_glo_phy-cur_nrt_0.25deg_P1M-m"]
SAT_IDS = sorted(SAT_IDS, key=lambda i: 0 if "_my_" in i else 1)
print("satellite current datasets:", SAT_IDS)

def sat_year(y):
    f = f"{C6}/sat_cur_sst_{y}.nc"
    if os.path.exists(f) and y <= this_year - 2: return xr.open_dataset(f).load()
    t0, t1 = pd.Timestamp(y, 1, 1), min(pd.Timestamp(y, 12, 31), TODAY)
    cur = []
    for sid in SAT_IDS:
        try:
            u = regrid(cm_open(sid, "uo", t0, t1, depth=(14, 16)).load()); v = regrid(cm_open(sid, "vo", t0, t1, depth=(14, 16)).load())
            u, v = [x.isel(depth=0, drop=True) if "depth" in x.dims else x for x in (u, v)]
            d = xr.Dataset({"u": u, "v": v}); d["time"] = month_start(d.time.values); cur.append(d)
        except Exception as e: pass
    if not cur: return None
    cur = clean_time(xr.concat(cur, "time"))                                     # reanalysed first, near-real-time after
    oi = ee.ImageCollection("NOAA/CDR/OISST/V2_1").select("sst")
    def sst_m(t):
        img = oi.filterDate(str(t.date()), str((t + pd.offsets.MonthBegin(1)).date())).mean().multiply(0.01).unmask(-9999, False).toFloat()
        return ee_grid(img, ["sst"])["sst"]
    months = pd.DatetimeIndex(cur.time.values)
    with ThreadPoolExecutor(max_workers=10) as ex: S = list(ex.map(sst_m, months))
    cur["sst"] = (("time", "lat", "lon"), np.stack(S))
    if y <= this_year - 2: cur.to_netcdf(f)
    return cur

SAT = [d for d in (sat_year(y) for y in range(Y0, this_year + 1)) if d is not None]
SAT = clean_time(xr.concat(SAT, "time")); print("satellite:", str(SAT.time.values[0])[:7], "->", str(SAT.time.values[-1])[:7])

def adv_terms(T, U, V):
    Ta, Ua, Va = anom(T), anom(U), anom(V)
    cm_ = lambda c: c.sel(month=T["time.month"]).drop_vars("month")
    Tbar, Ubar, Vbar = cm_(clim(T)), cm_(clim(U)), cm_(clim(V)); k = SEC_MONTH
    return {"zonal advective feedback": -Ua * ddx(Tbar) * k, "mean-current advection": (-Ubar * ddx(Ta) - Vbar * ddy(Ta)) * k,
            "meridional advection": -Va * ddy(Tbar) * k, "total horizontal advection anomaly": (anom(-U * ddx(T)) + anom(-V * ddy(T))) * k}
tt = np.intersect1d(SAT.time.values, G.time.values)
sat = adv_terms(SAT.sst.sel(time=tt), SAT.u.sel(time=tt), SAT.v.sel(time=tt))
glo = adv_terms(layer(G.thetao, H).sel(time=tt), layer(G.uo, H).sel(time=tt), layer(G.vo, H).sel(time=tt))
CMP = []
for name in BOXES:
    for kk in sat:
        s, g_ = box_mean(sat[kk], name), box_mean(glo[kk], name); j = pd.concat([s.rename("sat"), g_.rename("glorys")], axis=1).dropna()
        row = {"box": name, "term": kk, "r_monthly": j.sat.corr(j.glorys), "rmse": float(np.sqrt(((j.sat - j.glorys) ** 2).mean())), "n_months": len(j)}
        for y in EVENTS:
            sel = j[(j.index.year == y) & (j.index.month.isin(SEASON))]
            row[f"{y} sat"] = sel.sat.sum() if len(sel) >= 3 else np.nan; row[f"{y} GLORYS"] = sel.glorys.sum() if len(sel) >= 3 else np.nan
        CMP.append(row)
CMP = pd.DataFrame(CMP); CMP.round(3).to_csv(f"{OUT}/C_satellite_vs_glorys.csv", index=False); display(CMP.round(2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4.4))
for ax, name in zip(axs, BOXES):
    d = CMP[(CMP.box == name) & (CMP.term == "zonal advective feedback")].iloc[0]
    x = np.arange(len(EVENTS))
    ax.bar(x - .18, [d[f"{y} GLORYS"] for y in EVENTS], .36, color="#dc2626", label="GLORYS12 (0–50 m)")
    ax.bar(x + .18, [d[f"{y} sat"] for y in EVENTS], .36, color="#0ea5e9", label="satellite (GlobCurrent 15 m + OISST)")
    ax.axhline(0, c="k", lw=.8); ax.set_xticks(x); ax.set_xticklabels(EVENTS); ax.legend(fontsize=8)
    ax.set_title(f"{name}: zonal advective feedback, Mar–Sep (°C)\nmonthly r = {d.r_monthly:.2f}, n = {int(d.n_months)}", loc="left", fontweight="bold", fontsize=9)
plt.tight_layout()
for ext in ("png", "pdf"): plt.savefig(f"{OUT}/C_satellite_check.{ext}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
lines = ["# Notebook 06 summary — heat budget v2", f"_Run {TODAY:%d %b %Y}. GLORYS12 monthly + daily, ERA5 fluxes, GlobCurrent + OISST. Layer 0–{H} m; anomalies vs {CLIM[0]}–{CLIM[1]}._", ""]
for (name, y), r in main.iterrows():
    parts = ", ".join(f"{SHORT.get(c, c)} {r[c]:+.2f}" for c in TERMS + ["eddy′ (°C)"] if pd.notna(r[c]))
    lines.append(f"- **{name} {y}:** observed {r.tendency:+.2f} °C; {parts}; residual {r['residual after eddy']:+.2f} °C "
                 f"(closure {r['closure_after_%']:.0f} %, {r['closure_%']:.0f} % before eddies).")
for _, r in CMP[CMP.term == "zonal advective feedback"].iterrows():
    lines.append(f"- **Satellite check, {r.box}:** zonal advective feedback, monthly r = {r.r_monthly:.2f}; 2026 Mar–Sep: satellite {r['2026 sat']:+.2f} vs GLORYS {r['2026 GLORYS']:+.2f} °C.")
open(f"{OUT}/summary06.md", "w").write("\n".join(lines)); print("\n".join(lines)); print("\nSaved to", OUT)

## Reading the results
* **Thermocline vs upwelling:** compare the *thermocline feedback* (warmer water being upwelled) with the *Ekman feedback* (less upwelling). Notebook 04's two-term regression suggested the thermocline dominates in all four events; this budget tests it term by term.
* **Closure:** "closure %" is the share of the budget explained by resolved terms. Above roughly 70 % is a usable budget; the remaining residual is mostly vertical mixing and entrainment, which the reanalysis output does not provide.
* **Eddies:** tropical instability waves normally carry heat *into* the cold tongue (they warm it). El Niño suppresses them, so the eddy *anomaly* is expected to be a small **cooling** term (Jochum & Murtugudde 2006).
* **Satellite check:** GlobCurrent uses altimetry and an Ekman model, GLORYS assimilates altimetry, so the two are not fully independent; agreement shows the zonal advection term is not an artefact of the ocean model's physics.

**References:** An & Jin 2004 *J. Climate*; Jin, Kim & Bejarano 2006 *GRL* (Bjerknes stability index); Ren & Jin 2013 *J. Climate* (ENSO mixed-layer budget); Kug et al. 2009 *J. Climate*; Paulson & Simpson 1977 *JPO*; Jochum & Murtugudde 2006 *JPO* (TIW heat flux); Wang & McPhaden 2000 *J. Climate* (equatorial heat balance); Rio et al. 2014 *GRL* (GlobCurrent); Huang et al. 2021 *J. Climate* (OISST v2.1); Lellouche et al. 2021 *Front. Earth Sci.* (GLORYS12); Hersbach et al. 2020 *QJRMS* (ERA5).